In [ ]:
BASE_PATH = r"C:\Users\MEL\Downloads\DATA\SEMANA 4\Proyect_Lab\Proyecto_lab"

: 

In [ ]:
# Importamos las librerías necesarias y cargamos el dataset crudo
import pandas as pd
import numpy as np
import os



In [ ]:
filepath = r"C:\Users\MEL\Downloads\DATA\SEMANA 4\Proyect_Lab\Proyecto_lab\data\raw\trials_raw.csv"
df = pd.read_csv(filepath)
print(f"✅ Datos cargados: {df.shape}")
df.head()

In [ ]:
# Exploramos la estructura del dataset antes de limpiar
print("=== TIPOS DE DATOS ===")
print(df.dtypes)
print("\n=== VALORES NULOS ===")
print(df.isnull().sum())
print("\n=== ESTADÍSTICAS ===")
print(df.describe())

In [ ]:
# Técnica 1: Eliminar duplicados por nct_id
before = len(df)
df = df.drop_duplicates(subset=["nct_id"], keep="first")
after = len(df)
print(f"✅ Duplicados eliminados: {before - after}")
print(f"   Filas: {before} → {after}")

In [ ]:
# Técnica 2: Tratar valores nulos
df["enrollment"] = df["enrollment"].fillna(df["enrollment"].median())
text_cols = ["phase", "sponsor_class", "conditions", "countries"]
for col in text_cols:
    df[col] = df[col].fillna("Unknown")
print("✅ Nulos tratados")
print(df.isnull().sum())

In [ ]:
# Técnica 3: Estandarizar valores categóricos
phase_map = {
    "PHASE1": "Phase 1", "PHASE2": "Phase 2",
    "PHASE3": "Phase 3", "PHASE4": "Phase 4",
    "EARLY_PHASE1": "Phase 1 (Early)", "NA": "Not Applicable"
}
status_map = {
    "RECRUITING": "Recruiting", "COMPLETED": "Completed",
    "TERMINATED": "Terminated", "NOT_YET_RECRUITING": "Not Yet Recruiting",
    "ACTIVE_NOT_RECRUITING": "Active, Not Recruiting",
    "WITHDRAWN": "Withdrawn"
}
df["phase"] = df["phase"].map(phase_map).fillna(df["phase"])
df["status"] = df["status"].map(status_map).fillna(df["status"])
print("✅ Strings estandarizados")
print(df["phase"].value_counts())
print(df["status"].value_counts())

In [ ]:
# Técnica 4: Normalizar fechas y extraer año
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
df["start_year"] = df["start_date"].dt.year
df.loc[df["start_year"] < 1990, "start_year"] = np.nan
df.loc[df["start_year"] > 2025, "start_year"] = np.nan
print("✅ Fechas normalizadas")
print(df["start_year"].value_counts().sort_index().tail(10))

In [ ]:
# Técnica 5: Tratar outliers en enrollment
print(f"Antes — max: {df['enrollment'].max()}, media: {df['enrollment'].mean():.0f}")
df.loc[df["enrollment"] > 1_000_000, "enrollment"] = np.nan
df["enrollment"] = df["enrollment"].fillna(df["enrollment"].median())
df["enrollment"] = df["enrollment"].astype(int)
print(f"Después — max: {df['enrollment'].max()}, media: {df['enrollment'].mean():.0f}")
print("✅ Outliers tratados")

In [ ]:
# Técnica 6: Extraer país principal
df["primary_country"] = df["countries"].str.split(";").str[0].str.strip()
df["primary_country"] = df["primary_country"].replace("", "Unknown")
print("✅ País principal extraído")
print(df["primary_country"].value_counts().head(10))

In [ ]:
# Técnica 7: Limpiar espacios en columnas de texto
df["title"] = df["title"].str.strip()
df["sponsor"] = df["sponsor"].str.strip()
print("✅ Columnas limpias")
print(f"Shape final: {df.shape}")
df.head()

In [ ]:
# Guardar dataset limpio
BASE_PATH = r"C:\Users\MEL\Downloads\DATA\SEMANA 4\Proyect_Lab\Proyecto_lab"
df.to_csv(f"{BASE_PATH}\\data\\clean\\trials_clean.csv", index=False)
print(f"✅ Datos limpios guardados")
print(f"Shape final: {df.shape}")